In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

True
NVIDIA GeForce RTX 5050 Laptop GPU
13.0


In [19]:
import pandas as pd
df = pd.read_csv("b_hashed_list.csv")
df["Date of document:"] = (
    df["Date of document:"]
    .astype(str)
    .str.replace(r"[:;].*$", "", regex=True)
    .str.strip()
)
df.to_csv("b_hashed_list.csv")

In [ ]:
import pandas as pd

df = pd.read_csv(
    "b_hashed_list.csv",
    usecols=[
        "title",
        "ref",
        "status",
        "CELEX number:",
        "Author:",
        "Date of document:",
        "link",
        "Latest consolidated version:",
        "hash_id"
    ]
)
#clean metadata
df = df.set_index("hash_id")

In [ ]:
import os
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

# 1. Charger le texte
path = "node_modules/llamaindex/examples/abramov.txt"
with open(path, "r", encoding="utf-8") as f:
    essay = f.read()

# 2. Client Qdrant
client = QdrantClient(url="http://localhost:6333")

vector_store = QdrantVectorStore(
    client=client,
    collection_name="my_collection",   # nom de la collection
)

# 3. Stockage
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 4. Création du document
document = Document(text=essay, id_=path)

# 5. Construction de l’index
index = VectorStoreIndex.from_documents(
    [document],
    storage_context=storage_context
)

# 6. Query Engine
query_engine = index.as_query_engine()

# 7. Question
response = query_engine.query("What did the author do in college?")

print(response)


In [12]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from pathlib import Path

def add_file_metadata(path):
    p = Path(path)   
    hash_id = p.stem      
    row = df.loc[hash_id].to_dict()          
    return {"hash_id": hash_id, **row}

documents = SimpleDirectoryReader(
    "texts/",
    file_metadata=add_file_metadata).load_data()
print(type(documents))
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
print(type(splitter))
nodes = splitter.get_nodes_from_documents(documents)
print(type(nodes))

# 3. Embedding model (Hugging Face)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2", device = "cuda") #device="cpu"


# 4. Construction de l’index
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes, embed_model=embed_model)

# 5. Sauvegarde locale de l’index
index.storage_context.persist(persist_dir="./index_storage")

<class 'list'>
<class 'llama_index.core.node_parser.text.sentence.SentenceSplitter'>


ValueError: Metadata length (512) is longer than chunk size (512). Consider increasing the chunk size or decreasing the size of your metadata to avoid this.

In [14]:
import pandas as pd
from pathlib import Path

from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import os

load_dotenv()

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")


# 1. Charger le DataFrame filtré
df = pd.read_csv(
    "b_hashed_list.csv",
    usecols=[
        "title",
        "ref",
        "status",
        "CELEX number:",
        "Author:",
        "Date of document:",
        "link",
        "Latest consolidated version:",
        "hash_id"
    ]
)
df = df.set_index("hash_id")

In [16]:
print(qdrant_url)

https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/


In [17]:
# 2. Ajout métadonnées → Document
def add_file_metadata(path):
    p = Path(path)
    hash_id = p.stem
    row = df.loc[hash_id].to_dict()
    return {"hash_id": hash_id, **row}


# 3. Charger documents
documents = SimpleDirectoryReader(
    "texts/",
    file_metadata=add_file_metadata
).load_data()

splitter = SentenceSplitter(chunk_size=2048, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)


# 4. Embedding model
embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)


# 5. Qdrant setup
client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="eurlex"
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)


# 6. Construction de l’index (Qdrant)
index = VectorStoreIndex.from_documents(
    nodes,
    embed_model=embed_model,
    storage_context=storage_context
)
index.storage_context.persist(persist_dir="./index_storage")

2025-11-16 16:12:19,504 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2025-11-16 16:12:25,091 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/ "HTTP/1.1 200 OK"
2025-11-16 16:12:25,321 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/eurlex/exists "HTTP/1.1 200 OK"
2025-11-16 16:12:54,961 - INFO - HTTP Request: PUT https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/eurlex "HTTP/1.1 200 OK"
2025-11-16 16:12:55,333 - INFO - HTTP Request: PUT https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/eurlex/index?wait=true "HTTP/1.1 200 OK"
2025-11-16 16:12:55,416 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/eurlex "HTTP/1.1 200 OK"
2025-11-16 16:12:56,2

In [18]:
# 7. Query Engine
query_engine = index.as_query_engine()

response = query_engine.query("could you tell us the most important law?")
print(response)


ValueError: 
******
Could not load OpenAI model. If you intended to use OpenAI, please check your OPENAI_API_KEY.
Original error:
No API key found for OpenAI.
Please set either the OPENAI_API_KEY environment variable or openai.api_key prior to initialization.
API keys can be found or created at https://platform.openai.com/account/api-keys

******